# Demonstrating the Feasibility of Embedded AI for Capsule Endoscopy: 

# A TinyML Prototype for Real-Time Classification of Gastrointestinal Images

### Example notebook for the bachelor's thesis of Luis Schaich

The structure of this notebook was adapted from:

R. Berta, A. Dabbous, L. Lazzaroni, D. P. Pau and F. Bellotti, "Developing a TinyML Image Classifier in an Hour," in IEEE Open Journal of the Industrial Electronics Society, vol. 5, pp. 946-960, 2024, doi: 10.1109/OJIES.2024.3451959.

https://github.com/Elios-Lab/tice


## Import needed libraries

The following libraries need to be installed:

In [ ]:

#%pip install scikit-learn tensorflow keras keras-tuner numpy 
#%pip install matplotlib seaborn Pillow tabulate pandas lime scikit-image

For squeezenet: The file squeezenet.py must be located in the same working directory as this notebook

The squeezenet.py file could be found in the same github repository

In [ ]:
import sklearn.datasets
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.linear_model import SGDClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import time
import tracemalloc
import sklearn
from keras.losses import binary_crossentropy
from keras import layers, models, optimizers
import tensorflow as tf
from tensorflow import keras
import keras_tuner as kt
from keras_tuner.tuners import BayesianOptimization
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (confusion_matrix, precision_recall_fscore_support, roc_auc_score, matthews_corrcoef, roc_curve, auc)
from sklearn.preprocessing import label_binarize

from squeezenet import SqueezeNet_11
from PIL import Image
import io
import os
import random
import math
from pathlib import Path
import pathlib
from tabulate import tabulate
import pandas as pd
from sklearn.utils.class_weight import compute_class_weight
from lime import lime_image
from skimage.segmentation import mark_boundaries

# **Download dataset:**

The Kvasir-Capsule dataset can be downloaded from:
https://datasets.simula.no/kvasir-capsule/

The data should already be in the required folder layout

## Load data

Images need to be stored like this:
```text
labelled_images/
├── Angiectasia/
│   ├── 001.jpg
│   ├── 002.jpg
├── Polyp/
│   ├── 003.jpg
│   ├── 004.jpg
└── Ulcer/
    ├── 005.jpg
    ├──...
```

In [ ]:
DATA_DIR = Path(".../tinyML_project/data/kvasir_capsule/labelled_images")

In [ ]:
PLOT_DIR = ".../tinyML_project/plots/..."
os.makedirs(PLOT_DIR, exist_ok=True)

Those values could be adjusted to the specific needs


In [ ]:
img_size = (336, 336)
batch_size = 32
max_epochs=300

In [ ]:
class_names = sorted([p.name for p in DATA_DIR.iterdir() if p.is_dir()])
num_classes = len(class_names)
class_to_idx = {c: i for i, c in enumerate(class_names)}

Only .jpg files are accepted to avoid issues caused by auxiliary files such as thumbnail files

In [ ]:
valid_ext = {".jpg"}
file_paths = []
labels = []
for c in class_names:
    for fp in (DATA_DIR / c).rglob("*"):
        if fp.suffix.lower() in valid_ext:
            file_paths.append(str(fp))
            labels.append(class_to_idx[c])

file_paths = np.array(file_paths)
y = np.array(labels, dtype=np.int32)

## Check the number of samples:

In [ ]:
samples_number = len(file_paths)
print("Number of samples:", samples_number)
print("Classes:", class_names)

## Figure of one sample image for each class:

In [ ]:
K = len(class_names)

rows = 2
columns = math.ceil(K / rows)

fig = plt.figure(figsize=(4 * columns, 4 * rows))
for i in range(K):
    # number of images each class
    n_i = np.count_nonzero(y == i)
    idxs = np.where(y == i)[0]

    # random image for each class
    index = random.choice(idxs)
    img_path = file_paths[index]
    img = tf.keras.utils.load_img(img_path, target_size=img_size)
    title = f"{class_names[i]} (N={n_i})"

    ax = fig.add_subplot(rows, columns, i + 1)
    ax.imshow(img)
    ax.set_title(title,fontsize=17,pad=10, fontweight='heavy')
    ax.axis("off")

plt.tight_layout()
plt.savefig(
    os.path.join(PLOT_DIR, "samples_per_class.png"),
    dpi=300,
    bbox_inches="tight"
)
plt.show()

## Dataset split into training set, validation set and test set: 70%,15%,15%

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    file_paths, y, test_size=0.15, random_state=1, stratify=y
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.15, random_state=1, stratify=y_train_full
)

print("Splits:", "train:", len(X_train), "valid:", len(X_valid), "test:", len(X_test))

Class-weighted loss setup for imbalanced training labels:

In [ ]:
classes = np.unique(y_train)
cw_raw  = compute_class_weight(class_weight="balanced", classes=classes, y=y_train).astype(np.float32)
# Dampen and clip extreme weights
cw = np.sqrt(cw_raw)                 # soften extremes
cw = cw / np.mean(cw)                # normalize around 1.0
cw = np.clip(cw, 0.5, 3.0)           # prevent huge gradients

class_weight = {int(c): float(w) for c, w in zip(classes, cw)}
print("Class weights:", class_weight)

Data load: with resizing option and normalizing the values from 0.0, 255.0 to 0.0, 1.0

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def load_and_preprocess(path, label):
    img_bytes = tf.io.read_file(path)
    img = tf.image.decode_image(img_bytes, channels=3, expand_animations=False)
    img = tf.image.resize(img, img_size)
    img = tf.cast(img, tf.float32) / 255.0
    return img, label


Data augmentation:

In [ ]:
color_jitter = keras.layers.RandomColorJitter(value_range=(0.0, 1.0),brightness_factor=0.2,contrast_factor=0.2,saturation_factor=(0.4, 0.6),hue_factor=0.05)
gaussian_noise = keras.layers.GaussianNoise(0.1)
flip = keras.layers.RandomFlip('horizontal_and_vertical')
rotation = keras.layers.RandomRotation(0.18)
gaussian_blur = keras.layers.RandomGaussianBlur(factor=(0.2, 0.7), kernel_size=7, sigma=(0.7, 1.0), value_range=(0.0, 1.0))

In [ ]:
aug_copies_per_class = {
    0: 2.0,   # class id 0 -> +2 augmented copies per original
    1: 0.25,
    2: 0.5,
    3: 2.0,
    4: 0.5,
    5: 1.0,
    6: 0.25,
    7: 0.1,
    8: 0.25,
    9: 0.1,
    10: 2.0,
    11: 0.1,
    12: 0.1,
    13: 0.25,   
}
rng = np.random.default_rng(1)
aug_paths = []
aug_labels = []

for p, lbl in zip(X_train, y_train):
    lbl = int(lbl)
    c = float(aug_copies_per_class.get(lbl))
    full = int(np.floor(c))
    frac = c - full

    for _ in range(full):
        aug_paths.append(p)
        aug_labels.append(lbl)

    if rng.random() < frac:
        aug_paths.append(p)
        aug_labels.append(lbl)

X_aug = np.array(aug_paths)
y_aug = np.array(aug_labels, dtype=np.int32)

def apply_aug(img, label):
    idx = tf.random.uniform([], minval=0, maxval=6, dtype=tf.int32) # 0 = no aug, 1..5 = only one augmentation

    branches = [
        lambda: img,
        lambda: color_jitter(img, training=True),
        lambda: gaussian_noise(img, training=True),
        lambda: flip(img, training=True),
        lambda: rotation(img, training=True),
        lambda: gaussian_blur(img, training=True),
    ]
    img = tf.switch_case(idx, branch_fns=branches)
    img = tf.clip_by_value(img, 0.0, 1.0)
    return img, label

In [ ]:
# Original train samples (unchanged)
train_base = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_base = train_base.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)

# Extra augmented samples (class-dependent amount)
train_aug = tf.data.Dataset.from_tensor_slices((X_aug, y_aug))
train_aug = train_aug.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
train_aug = train_aug.map(apply_aug, num_parallel_calls=AUTOTUNE)

train_ds = train_base.concatenate(train_aug)
train_ds = train_ds.shuffle(len(X_train) + len(X_aug), seed=1, reshuffle_each_iteration=True)
train_ds = train_ds.batch(batch_size).prefetch(AUTOTUNE)

#Untouched validation and test images:

valid_ds = tf.data.Dataset.from_tensor_slices((X_valid, y_valid))
valid_ds = valid_ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
valid_ds = valid_ds.batch(batch_size).prefetch(AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))
test_ds = test_ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
test_ds = test_ds.batch(batch_size).prefetch(AUTOTUNE)

Visualizes the amount of added images with data augmentation:

In [ ]:
train_counts_before = np.bincount(y_train, minlength=num_classes)
added_counts = np.bincount(y_aug, minlength=num_classes)
train_counts_after_nominal = train_counts_before + added_counts

total_before = int(train_counts_before.sum())
total_after_nominal = int(train_counts_after_nominal.sum())
virtual_new_samples = total_after_nominal - total_before

print(f"Train samples before augmentation: {total_before}")
print(f"Nominal train samples after augmentation (per epoch): {total_after_nominal}")
print(f"Virtual additional samples per epoch: {virtual_new_samples}")

summary_df = pd.DataFrame({
    "class_id": np.arange(num_classes, dtype=np.int32),
    "class_name": class_names,
    "train_count_before": train_counts_before,
    "added_augmented_samples": added_counts,
    "train_count_after_nominal": train_counts_after_nominal,
})
print(summary_df.sort_values("class_id").to_string(index=False))

One visualization for each augmentation layer + untouched image

In [ ]:
def _load_train_image_by_index(idx):
    path = tf.constant(X_train[idx])
    label = tf.constant(y_train[idx], dtype=tf.int32)
    img, _ = load_and_preprocess(path, label)
    return img


example_idx = int(np.random.randint(0, len(X_train)))
base_img = _load_train_image_by_index(example_idx)

panel_images = [
    ('Untouched', base_img),
    ('RandomColorJitter', color_jitter(base_img, training=True)),
    ('GaussianNoise', gaussian_noise(base_img, training=True)),
    ('RandomFlip', flip(base_img, training=True)),
    ('RandomRotation', rotation(base_img, training=True)),
    ('RandomGaussianBlur', gaussian_blur(base_img, training=True))
]

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for ax, (title, img) in zip(axes.flatten(), panel_images):
    ax.imshow(tf.clip_by_value(img, 0.0, 1.0).numpy())
    ax.set_title(title, fontsize=11)
    ax.axis('off')

plt.suptitle('Data augmentation preview', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
height, width = img_size
channels = 3
number_of_channels = 3
print(width, height, number_of_channels)

To save the splits as .npz

In [ ]:
split_dir = Path(".../tinyML_project/data/kvasir_capsule/splits/...")
split_dir.mkdir(parents=True, exist_ok=True)

# Save splits
np.savez(
    split_dir / "split.npz",
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
    X_test=X_test,
    y_test=y_test
)

print("Dataset split saved successfully")

To save only the test split

In [ ]:
split_dir = Path(".../tinyML_project/data/kvasir_capsule/splits/...")
split_dir.mkdir(parents=True, exist_ok=True)

# Save splits
np.savez(
    split_dir / "test_only.npz",
    X_test=X_test,
    y_test=y_test
)

print("Dataset split saved successfully")

In [ ]:
CNN_classifier_accuracy=[]
NN_model_sizes=[]

# **CNN models**
This section contains the implementation of the general helper functions

In [ ]:
y_test = np.concatenate([y.numpy() for _, y in test_ds], axis=0)

In [ ]:
def get_best_epoch(history, monitor="val_loss", mode="min"):
    values = history.history.get(monitor, None)
    if values is None:
        raise ValueError(f"Monitor '{monitor}' not found in history. Available keys: {list(history.history.keys())}")
    if mode == "max":
        best_idx = int(np.argmax(values))
    elif mode == "min":    
        best_idx = int(np.argmin(values))
    else:
        raise ValueError("mode must be 'max' or 'min'")

    best_epoch = best_idx + 1  
    best_value = float(values[best_idx])
    return best_epoch, best_value

In [ ]:
def NN_hyperparameter_fit(hyperparameter,Type_model,
                          model_name=None,
                          monitor="val_loss",
                          mode="min",              
                          ):
  if (Type_model=="SqueezeNet"):
    model=build_squeezenet_11(hyperparameter) 
  model.summary()
  early = tf.keras.callbacks.EarlyStopping(
        monitor=monitor,
        mode=mode,
        patience=15,
        min_delta=1e-4,
        restore_best_weights=True,
        verbose=1)
  callbacks = []
  callbacks.append(early)
  history = model.fit(train_ds, epochs=max_epochs, batch_size=batch_size, validation_data=valid_ds, class_weight=class_weight, callbacks=callbacks, verbose=1)
  best_epoch, best_value = get_best_epoch(history, monitor=monitor, mode=mode)
  print(f"[{model_name}] best_epoch={best_epoch}, best_{monitor}={best_value:.4f}")
  plot_model_performance(history,len(history.history['loss']), model_name)
  return model,history, best_epoch, best_value



#### 1. Weighted evaluation

Calculates precision, recall, and F1-score for each class and then averages them using class support as weights. This gives more influence to classes with more samples and is therefore suitable for imbalanced datasets.

In [ ]:
def evaluate_NN_models_weighted(model_list):
  loss, accuracy , y_pred ,precision, recall ,f1_score ,support ,confusion_matrix_list, mcc_scores, auc_scores, latencies, fps_values = [], [], [], [], [], [], [], [], [], [], [], []
  # Prepare labels for ROC-AUC
  classes = np.unique(y_test)
  y_test_bin = label_binarize(y_test, classes=classes)
  model_idx = 0
  for model in model_list:
    loss_value,accuracy_value=model.evaluate(test_ds)
    loss.append(loss_value)
    accuracy.append(accuracy_value)

    y_proba = model.predict(test_ds, verbose=0)
    y_pred_value=np.argmax(y_proba, axis = 1)
    y_pred.append(y_pred_value)

    precision_value, recall_value, f1_score_value, support_value = sklearn.metrics.precision_recall_fscore_support(y_test , y_pred_value, average= 'weighted' )
    precision.append(precision_value)
    recall.append(recall_value)
    f1_score.append(f1_score_value)
    support.append(support_value)
    #Confusion matrix:
    confusion_matrix_value=confusion_matrix(y_test, y_pred_value)
    confusion_matrix_list.append(confusion_matrix_value)
    # Matthews Correlation Coefficient
    mcc = matthews_corrcoef(y_test, y_pred_value)
    mcc_scores.append(mcc)
    # ROC-AUC (macro)
    auc_value = roc_auc_score(y_test_bin, y_proba, average="macro", multi_class="ovr")
    auc_scores.append(auc_value)
    #ROC plot:
    plt.figure(figsize=(8, 6))
    for i in range(len(classes)):
      fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
      roc_auc = auc(fpr, tpr)
      plt.plot(fpr, tpr, label=f"{class_names[i]} (AUC={roc_auc:.2f})")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    model_name = names_models[model_idx]
    plt.title(f"ROC Curve Model {model_name}")
    model_idx += 1
    plt.legend()
    plt.grid(True)
    plt.savefig(
      os.path.join(PLOT_DIR, f"ROC_Curve_Model_{model_name}.png"),
      dpi=300,
      bbox_inches="tight"
    )
    plt.show()
    
    #Latency & FPS:
    x0, _ = next(iter(test_ds.unbatch().batch(1)))
    start = time.time()
    _ = model.predict(x0, verbose=0)
    end = time.time()
    latency = (end - start)  # seconds per inference
    latencies.append(latency)
    if latency < 1e-6:
      fps_values.append("x")
    else:
      fps_values.append(1.0 / latency)
    
    
  return [accuracy ,precision, recall ,f1_score, mcc_scores, auc_scores, latencies, fps_values],confusion_matrix_list,y_pred


#### 2. Macro evaluation

Calculate metrics for each label, and find their unweighted mean. This does not take label imbalance into account

In [ ]:
def evaluate_NN_models_macro(model_list):
  loss, accuracy , y_pred ,precision, recall ,f1_score ,support  = [], [], [], [], [], [], []
  
  for model in model_list:
    loss_value,accuracy_value=model.evaluate(test_ds)
    loss.append(loss_value)
    accuracy.append(accuracy_value)

    y_proba = model.predict(test_ds, verbose=0)
    y_pred_value=np.argmax(y_proba, axis = 1)
    y_pred.append(y_pred_value)

    precision_value, recall_value, f1_score_value, support_value = sklearn.metrics.precision_recall_fscore_support(y_test , y_pred_value, average= 'macro' )
    precision.append(precision_value)
    recall.append(recall_value)
    f1_score.append(f1_score_value)
    support.append(support_value)
  return [accuracy ,precision, recall ,f1_score],y_pred

### 3. Micro evaluation

Calculate metrics globally by counting the total true positives, false negatives and false positives.

In [ ]:
def evaluate_NN_models_micro(model_list):
  loss, accuracy , y_pred ,precision, recall ,f1_score ,support  = [], [], [], [], [], [], []
  
  for model in model_list:
    loss_value,accuracy_value=model.evaluate(test_ds)
    loss.append(loss_value)
    accuracy.append(accuracy_value)

    y_proba = model.predict(test_ds, verbose=0)
    y_pred_value=np.argmax(y_proba, axis = 1)
    y_pred.append(y_pred_value)

    precision_value, recall_value, f1_score_value, support_value = sklearn.metrics.precision_recall_fscore_support(y_test , y_pred_value, average= 'micro' )
    precision.append(precision_value)
    recall.append(recall_value)
    f1_score.append(f1_score_value)
    support.append(support_value)
  
  return [accuracy ,precision, recall ,f1_score],y_pred

### 4. Per-class evaluation

In [ ]:
def evaluate_NN_models_None(model_list):
  loss, accuracy , y_pred ,precision, recall ,f1_score ,support  = [], [], [], [], [], [], []
  
  for model in model_list:
    loss_value,accuracy_value=model.evaluate(test_ds)
    loss.append(loss_value)
    accuracy.append(accuracy_value)

    y_proba = model.predict(test_ds, verbose=0)
    y_pred_value=np.argmax(y_proba, axis = 1)
    y_pred.append(y_pred_value)

    precision_value, recall_value, f1_score_value, support_value = sklearn.metrics.precision_recall_fscore_support(y_test , y_pred_value, average=None)
    precision.append(precision_value)
    recall.append(recall_value)
    f1_score.append(f1_score_value)
    support.append(support_value)

  return [accuracy ,precision, recall ,f1_score],y_pred

In [ ]:
def plot_model_performace_accuracy(history, model_name):
  fig, ax = plt.subplots(figsize=(7,5))
  ax.plot(history.history['accuracy'], label='Training Accuracy')
  ax.plot(history.history['val_accuracy'], label='Validation Accuracy')
  ax.set_title(f"Training and Validation ({model_name})")
  ax.set_xlabel("Epoch")
  ax.set_ylabel("Accuracy")
  ax.set_ylim(0.0, 1.01)
  ax.grid(True)
  ax.legend()
  fig.savefig(os.path.join(PLOT_DIR, f"Acc_{model_name}.png"), dpi=300, bbox_inches="tight")
  plt.show()

In [ ]:
def plot_model_performance(history,number_of_epochs, model_name):
  print('CNN model training accuracy: ', history.history['accuracy'][number_of_epochs - 1])
  print('CNN model validation accuracy: ', history.history['val_accuracy'][number_of_epochs - 1])
  plt.figure()
  plt.plot(history.history['loss'], 'g', label='Training Loss')
  plt.plot(history.history['val_loss'], color='orange', label='Validation Loss')
  plt.title('Training and Validation Loss')
  plt.ylabel('Loss')
  plt.xlabel('Epoch')
  plt.legend()
  plt.grid(True)
  plt.tight_layout()
  plt.savefig(os.path.join(PLOT_DIR, f"Loss_{model_name}.png"), dpi=300, bbox_inches="tight")

In [ ]:
def plot_confusion_matrix(ax, conf_matrix, title, cmap, class_names=None):
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap=cmap, cbar=False, ax=ax, 
                xticklabels=class_names if class_names is not None else True,
                yticklabels=class_names if class_names is not None else True)
    ax.set_title(title)
    ax.set_xlabel("Predicted Labels")
    ax.set_ylabel("True Labels")
    # Labels readable
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right",rotation_mode="anchor")
    ax.set_yticklabels( ax.get_yticklabels(), rotation=0)

def draw_confusion_matrix(matrix,name, class_names=None):
  colormap=["Blues","Greens","Oranges"]
  fig, axes = plt.subplots(1, 3, figsize=(15, 5))
  for i in range(len(matrix)):
    plot_confusion_matrix(axes[i], matrix[i], f"{name}{i+1}", colormap[i % len(colormap)], class_names=class_names)
  plt.tight_layout()
  plt.savefig(
      os.path.join(PLOT_DIR, f"Confusion_Matrix_{name}.png"),
      dpi=300,
      bbox_inches="tight"
    )
  plt.show()

In [ ]:
def print_table(data,headers):
  table_data = list(zip(*data))
  table = tabulate(table_data, headers=headers, tablefmt='grid')
  print(table)

In [ ]:
def convert_params_to_list(params_all):
  all_keys = set()
  for d in params_all:
      all_keys.update(d.keys())

  data_values = [[d.get(key, None) for d in params_all] for key in all_keys]
  return data_values,all_keys

In [ ]:
def format_memory(memory_bytes):
    if memory_bytes < 1024:
        return f"{memory_bytes} B"
    elif memory_bytes < 1024 * 1024:
        return f"{memory_bytes / 1024:.2f} KB"
    elif memory_bytes < 1024 * 1024 * 1024:
        return f"{memory_bytes / (1024 * 1024):.2f} MB"
    else:
        return f"{memory_bytes / (1024 * 1024 * 1024):.2f} GB"

To export tables:

In [ ]:
def export_table_to_files(data, headers, filename_base):
    rows = list(zip(*data)) 
    df = pd.DataFrame(rows, columns=headers)
    df.to_csv(f".../tinyML_project/evaluation/.../{filename_base}.csv", index=False)

    return df

# **CNN**

In [ ]:
model_names_CNN = [
    "SqueezeNet model 1",
    "SqueezeNet model 2",
    "SqueezeNet model 3"
]

## SqueezeNet

Other architectures could also be used here 

Enter here your values and number of classes:

In [ ]:
def build_squeezenet_11(hp):

    # Enter your learning rates 
    hp_learning_rate = hp.Choice('learning_rate', values=[5e-5,9e-5,1e-4,2e-4, 5e-4])
    # Tune the optimizer (Adam optimizer used)
    hp_optimizer = tf.keras.optimizers.Adam(learning_rate=hp_learning_rate)
    # Tune the number of filters in the Conv2D layer
    hp_filters = hp.Int('filters', min_value=16, max_value=128, step=16)
    compression = hp_filters / 64.0  # baseline 64 in SqueezeNet_11
    # Tune the dropout values
    dropout_rate = hp.Choice('dropout', values=[0.0,0.3,0.5])
    # Tuning weights initialization
    kernel_initializer = hp.Choice('kernel_initializer', values=['glorot_uniform'])


    model_SqueezeNet = SqueezeNet_11(
        input_shape=(height, width, number_of_channels),
        nb_classes=14,
        dropout_rate=dropout_rate,
        compression=compression,
        kernel_initializer=kernel_initializer

    )

    model_SqueezeNet.compile(
        optimizer=hp_optimizer,
        loss="sparse_categorical_crossentropy", 
        metrics=["accuracy"]
    )

    return model_SqueezeNet



## Observe the best-performing models and identify their hyperparameters:
### Give a new project_name for a new approach!!
### Hyperparameter search progress is saved in project_name

In [ ]:
# Bayesian approach to conduct the search
tuner_SqueezeNet = BayesianOptimization(
    build_squeezenet_11,
    objective='val_accuracy',
    max_trials=20,
    directory='tuner_single_dense',
    #IMPORTANT: change the project name for a new apprach!
    project_name='training_tuner_single_dense_RUN1'
)

#Training models with different hyperparameters.
tuner_SqueezeNet.search(train_ds, epochs=30, batch_size=batch_size, validation_data=valid_ds, class_weight=class_weight)


Gets the three best settings

In [ ]:
num_trials = 3
best_hp = tuner_SqueezeNet.get_best_hyperparameters(num_trials=num_trials)

for idx, hyperparameters in enumerate(best_hp):
    print(f"Set {idx + 1}: {hyperparameters.values}")

In [ ]:
best_hyperparameter_SqueezeNet=[best_hp[0],best_hp[1],best_hp[2]]

## Train the models and plot performance

In [ ]:
best_epochs = []
best_values = []

In [ ]:
# Build the model with the best hp.
model1_SqueezeNet,results1_SqueezeNet,be,bv=NN_hyperparameter_fit(best_hyperparameter_SqueezeNet[0],"SqueezeNet", model_name="SqueezeNet model 1",
                                                                 monitor="val_loss", mode="min")
best_epochs.append(be); best_values.append(bv)

In [ ]:
# Build the model with the second best hp.
model2_SqueezeNet,results2_SqueezeNet,be,bv=NN_hyperparameter_fit(best_hyperparameter_SqueezeNet[1],"SqueezeNet",model_name="SqueezeNet model 2",
                                                    monitor="val_loss", mode="min")
best_epochs.append(be); best_values.append(bv)

In [ ]:
# Build the model with the third best hp.
model3_SqueezeNet,results3_SqueezeNet,be,bv=NN_hyperparameter_fit(best_hyperparameter_SqueezeNet[2],"SqueezeNet",model_name="SqueezeNet model 3",
                                                    monitor="val_loss", mode="min")
best_epochs.append(be); best_values.append(bv)

In [ ]:
plot_model_performace_accuracy(results1_SqueezeNet, "SqueezeNet model 1")
plot_model_performace_accuracy(results2_SqueezeNet, "SqueezeNet model 2")
plot_model_performace_accuracy(results3_SqueezeNet, "SqueezeNet model 3")


Adding all models in a list:

In [ ]:
models_list_CNN = []
models_list_CNN.append(model1_SqueezeNet)
models_list_CNN.append(model2_SqueezeNet)
models_list_CNN.append(model3_SqueezeNet)

Save the models (.keras)

In [ ]:
model_files = []
model_size_kb = []
save_dir = ".../tinyML_project/models/..."
os.makedirs(save_dir, exist_ok=True)

for i, model in enumerate(models_list_CNN, start=1):
    model_path = f"{save_dir}/SqueezeNet_model_{i}.keras"
    model.save(model_path)
    print(f"Saved: {model_path}")

    model_files.append(model_path)
    size_kb = os.path.getsize(model_path) / 1024
    model_size_kb.append(round(size_kb, 2))

print("Model sizes (KB):", model_size_kb)

## Evaluation on the test set
### 1. Weighted (all parameters)

In [ ]:
names_models = ["SqueezeNet model 1", "SqueezeNet model 2", "SqueezeNet model 3"]

In [ ]:
result_all_models_CNN,confusion_matrix_CNN,predicitions=evaluate_NN_models_weighted(models_list_CNN)

headers=['models','hyperparameter','accuracy','precision','recall','f1_score','mcc', 'auc','latency', 'fps','model size KB']
data_normal_CNN=[names_models]

hyperparameters_values=[]
hyperparameters_values.append(hyperparameters.values)
data_normal_CNN.append(hyperparameters_values)
data_normal_CNN.extend(result_all_models_CNN)
data_normal_CNN.append(model_size_kb)
print_table(data_normal_CNN,headers)
CNN_classifier_accuracy=result_all_models_CNN[0]
df_cnn = export_table_to_files(data_normal_CNN, headers, filename_base="results_CNN_weighted")


Draw confusion matrix

In [ ]:
draw_confusion_matrix(confusion_matrix_CNN[0:3],"SqueezeNet model", class_names=class_names)

### 2. Macro (only accuracy, precision, recall and F1)

In [ ]:
result_macro_models_CNN,predicitions=evaluate_NN_models_macro(models_list_CNN)

headers=['models','accuracy','precision','recall','f1_score']
data_normal_CNN=[model_names_CNN]


data_normal_CNN.extend(result_macro_models_CNN)
print_table(data_normal_CNN,headers)
CNN_classifier_accuracy=result_all_models_CNN[0]
df_cnn = export_table_to_files(data_normal_CNN, headers, filename_base="results_CNN_macro")

### 3. Micro (only accuracy, precision, recall and F1)

In [ ]:
result_micro_models_CNN,predicitions=evaluate_NN_models_micro(models_list_CNN)

headers=['models','accuracy','precision','recall','f1_score']
data_normal_CNN=[model_names_CNN]

data_normal_CNN.extend(result_micro_models_CNN)
print_table(data_normal_CNN,headers)
CNN_classifier_accuracy=result_all_models_CNN[0]
df_cnn = export_table_to_files(data_normal_CNN, headers, filename_base="results_CNN_micro")

### 4. None -> per class (only accuracy, precision, recall and F1)

In [ ]:
result_None_models_CNN,predicitions=evaluate_NN_models_None(models_list_CNN)

headers=['models','accuracy','precision','recall','f1_score']
data_normal_CNN=[model_names_CNN]

data_normal_CNN.extend(result_None_models_CNN)
print_table(data_normal_CNN,headers)
CNN_classifier_accuracy=result_all_models_CNN[0]
df_cnn = export_table_to_files(data_normal_CNN, headers, filename_base="results_CNN_None")


# **TensorFlow Lite Model**


## Generalized functions for  TensorFlow Lite / Quantized Models

In [ ]:
def evaluate(model_file, ds, categoricalAccuarcy=True, max_batches=None):
    if(categoricalAccuarcy):
      accuracy = tf.keras.metrics.SparseCategoricalAccuracy()
    else:
      accuracy = tf.keras.metrics.BinaryAccuracy()

    interpreter = tf.lite.Interpreter(model_path=model_file)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    input_shape = input_details["shape"]
    target_h, target_w = input_shape[1], input_shape[2]

    y_preds = []  # predicted class index
    y_real = []   # true labels
    y_proba = []  #class probabilities
    for b, (x_batch, y_batch) in enumerate(ds):
        if max_batches is not None and b >= max_batches:
            break
        
        x_batch = tf.convert_to_tensor(x_batch)
        y_batch = tf.convert_to_tensor(y_batch)

        for x, y_true in zip(x_batch, y_batch):
            # resize input automatically
            x = tf.image.resize(x, (target_h, target_w))
            x_np = x.numpy()

            if input_details['dtype'] == np.uint8:
                input_scale, input_zero_point = input_details["quantization"]
                x = x / input_scale + input_zero_point

            x = np.expand_dims(x, axis=0).astype(input_details["dtype"])
            interpreter.set_tensor(input_details["index"], x)
            interpreter.invoke()

            y_pred = interpreter.get_tensor(output_details["index"])[0]
            accuracy.update_state(y_true, y_pred)

            if categoricalAccuarcy:
                y_preds.append(np.argmax(y_pred))
            else:
                y_preds.append(np.round(y_pred[0]))

            y_real.append(y_true)
            y_proba.append(y_pred)

    return accuracy.result(), y_preds, y_real, y_proba


#### 1. Weighted TFLite evaluation

In [ ]:
def evaluate_tflite_quantized_models_weighted(tflite_model_files, ds):
    tflite_model_accuracy = []
    tflite_model_predictions = []
    confusion_tflite_matrix_list = []
    precisions = []
    recalls = []
    f1_scores = []
    mcc_scores = []
    auc_scores = []
    latencies = []
    fps_values = []
    
    model_idx = 0
    

    for path in tflite_model_files:
        print("Evaluating:", path)

        accuracy, y_pred, y_real, y_proba = evaluate(path, ds, True)
        y_proba = np.array(y_proba)

        classes = np.unique(y_real)
        y_real_bin = label_binarize(y_real, classes=classes)

        tflite_model_accuracy.append(float(accuracy.numpy()))
        tflite_model_predictions.append(y_pred)
        
        

        precision_value, recall_value, f1_score_value, _ = sklearn.metrics.precision_recall_fscore_support(
            y_real, y_pred, average='weighted'
        )
        precisions.append(precision_value)
        recalls.append(recall_value)
        f1_scores.append(f1_score_value)

        confusion_matrix_value = confusion_matrix(y_real, y_pred)
        confusion_tflite_matrix_list.append(confusion_matrix_value)
        # Matthews Correlation Coefficient
        mcc = matthews_corrcoef(y_real, y_pred)
        mcc_scores.append(mcc)

        # ROC-AUC (macro)
        auc_value = roc_auc_score(y_real_bin, y_proba, average="macro", multi_class="ovr")
        auc_scores.append(auc_value)
        #ROC plot
        plt.figure(figsize=(8, 6))
        for i in range(len(classes)):
            fpr, tpr, _ = roc_curve(y_real_bin[:, i], y_proba[:, i])
            roc_auc = auc(fpr, tpr)
            plt.plot(fpr, tpr, label=f"{class_names[i]} (AUC={roc_auc:.2f})")
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        model_name = model_names_CNN[model_idx]
        plt.title(f"ROC Curve tflite {model_name} ")
        model_idx += 1
        plt.legend()
        plt.grid(True)
        plt.savefig(
            os.path.join(PLOT_DIR, f"ROC_Curve_TFLite_{model_name}.png"),
            dpi=300, bbox_inches="tight"
        )
        plt.show()
        # Latency & FPS
        start = time.time()
        _ = evaluate(path, ds, categoricalAccuarcy=True, max_batches=1)
        end = time.time()

        batch_latency = (end - start)  # seconds per inference
        latency = batch_latency/32.0 
        latencies.append(latency)
        if batch_latency < 1e-6:
            fps_values.append("x")
        else:
            fps_values.append(32.0 / batch_latency)
    
    return (
        [tflite_model_accuracy, precisions, recalls, f1_scores, mcc_scores, auc_scores, latencies, fps_values],
        confusion_tflite_matrix_list,
        tflite_model_predictions
    )

#### 2. Macro TFLite evaluation

In [ ]:
def evaluate_tflite_quantized_models_macro(tflite_model_files, ds):
    tflite_model_accuracy = []
    tflite_model_predictions = []
    precisions = []
    recalls = []
    f1_scores = []
    
    for path in tflite_model_files:
        print("Evaluating:", path)

        accuracy, y_pred, y_real, _ = evaluate(path, ds, True)

        tflite_model_accuracy.append(float(accuracy.numpy()))
        tflite_model_predictions.append(y_pred)

        precision_value, recall_value, f1_score_value, _ = sklearn.metrics.precision_recall_fscore_support(
            y_real, y_pred, average='macro'
        )
        precisions.append(precision_value)
        recalls.append(recall_value)
        f1_scores.append(f1_score_value)

        

    return (
        [tflite_model_accuracy, precisions, recalls, f1_scores],
        tflite_model_predictions
    )

#### 3. Micro TFLite evaluation

In [ ]:
def evaluate_tflite_quantized_models_micro(tflite_model_files, ds):
    tflite_model_accuracy = []
    tflite_model_predictions = []
    precisions = []
    recalls = []
    f1_scores = []
    
    for path in tflite_model_files:
        print("Evaluating:", path)

        accuracy, y_pred, y_real, _ = evaluate(path, ds, True)

        tflite_model_accuracy.append(float(accuracy.numpy()))
        tflite_model_predictions.append(y_pred)

        precision_value, recall_value, f1_score_value, _ = sklearn.metrics.precision_recall_fscore_support(
            y_real, y_pred, average='micro'
        )
        precisions.append(precision_value)
        recalls.append(recall_value)
        f1_scores.append(f1_score_value)

        

    return (
        [tflite_model_accuracy, precisions, recalls, f1_scores],
        tflite_model_predictions
    )

#### 4. Per-class TFLite evaluation

In [ ]:
def evaluate_tflite_quantized_models_None(tflite_model_files, ds):
    tflite_model_accuracy = []
    tflite_model_predictions = []
    precisions = []
    recalls = []
    f1_scores = []
    
    for path in tflite_model_files:
        print("Evaluating:", path)

        accuracy, y_pred, y_real, _ = evaluate(path, ds, True)

        tflite_model_accuracy.append(float(accuracy.numpy()))
        tflite_model_predictions.append(y_pred)

        precision_value, recall_value, f1_score_value, _ = sklearn.metrics.precision_recall_fscore_support(
            y_real, y_pred, average=None
        )
        precisions.append(precision_value)
        recalls.append(recall_value)
        f1_scores.append(f1_score_value)

        

    return (
        [tflite_model_accuracy, precisions, recalls, f1_scores],
        tflite_model_predictions
    )

## Converting all models to tflite model using tensorflow lite converter

In [ ]:
models_list=[model1_SqueezeNet, model2_SqueezeNet, model3_SqueezeNet]

In [ ]:
tflite_model_files = []
for index, model in enumerate(models_list):
    converter = tf.lite.TFLiteConverter.from_keras_model(models_list[index])
    
    tflite_model = converter.convert()
    
    # save to disk
    filename = f".../tinyML_project/converted_models/.../tflite_model/SqueezeNet_model_{index+1}.tflite"
    with open(filename, "wb") as f:
        f.write(tflite_model)
    
    tflite_model_files.append(filename)

print("Saved TFLite models:", tflite_model_files)
tflite_model_size = []   # global list

for file_path in tflite_model_files:
    size_kb = os.path.getsize(file_path) / 1024
    tflite_model_size.append(round(size_kb, 2))

print("TFLite model sizes (KB):", tflite_model_size)

## Evaluation:

Just weighted to compare with normal model -> see if anything has changed

In [ ]:
names_models = ['tflite SqueezeNet model 1','tflite SqueezeNet model 2', 'tflite SqueezeNet model 3']

In [ ]:
tflite_all_results, confusion_tflite_all, tflite_predictions = \
    evaluate_tflite_quantized_models_weighted(tflite_model_files, test_ds)
## create a table with all tf lite models results
headers=['model','hyperparameter','accuracy','precision','recall','f1_score','mcc', 'auc','latency', 'fps','model size KB']


data_tflite=[names_models]
hyperparameters_values=[]
hyperparameters_values.append(hyperparameters.values)
data_tflite.append(hyperparameters_values)
data_tflite.extend(tflite_all_results)
data_tflite.append(tflite_model_size)
print_table(data_tflite,headers)
NN_model_sizes=tflite_model_size
df_tflite = export_table_to_files(data_tflite, headers, filename_base="results_tflite_weighted")



Draw confusion matrix

In [ ]:
draw_confusion_matrix(confusion_tflite_all[0:3],"SqueezeNet Lite model", class_names=class_names)

# **Post-training quantization**

We can enable the default optimizations flag to quantize all fixed parameters (weights and biases) to 8-bit integers. Notice that scale and zero point for weights and bias can be calculated before the inference, because their ranges are already available.


## **Dynamic Quantization**

Dynamic quantization: To calculate scale and zero point for activations are calculated on-the-fly (online during inference). This means that the activations are always stored in float 32 and they are converted to integers while processing and back to floating point after the processing is done.

In [ ]:
tflite_dynamic_quantized_models = []

for i in range (0 , len(models_list)):
  converter = tf.lite.TFLiteConverter.from_keras_model(models_list[i])
  converter.optimizations = [tf.lite.Optimize.DEFAULT]
  tflite_dynamic_quantized_models.append(converter.convert())

In [ ]:
tflite_models_dir = pathlib.Path(".../tinyML_project/converted_models/...")

In [ ]:
tflite_dynamic_quantized_model_files = []
tflite_dynamic_quantized_model_size = []

base_filename_dynamic = 'tflite_dynamic_quantized_model'
# Create subfolder inside "converted_models"
path = tflite_models_dir/base_filename_dynamic

if not os.path.exists(path):
  os.makedirs(path)

for i in range (1, len(models_list) + 1):
  file_name =  f"{base_filename_dynamic}_{i}.tflite"
  tflite_dynamic_quantized_model_files.append(path/file_name)
  tflite_dynamic_quantized_model_files[i-1].write_bytes(tflite_dynamic_quantized_models[i-1])
  tflite_dynamic_quantized_model_size.append(os.path.getsize(tflite_dynamic_quantized_model_files[i - 1]) / float(2**10))
  print("TFlite dynamic quantized model in KB:", tflite_dynamic_quantized_model_size[i-1])

# Evaluation of the Dynamic Quantized Lite models

## 1. Weighted metrics

In [ ]:
names_models = ['Dynamic quantized SqueezeNet model 1','Dynamic quantized SqueezeNet model 2', 'Dynamic quantized SqueezeNet model 3']

In [ ]:
dynamic_quantized_all_results, confusion_dynamic_quantized_all, dynamic_quantized_predictions = \
    evaluate_tflite_quantized_models_weighted(tflite_dynamic_quantized_model_files, test_ds)

headers=['models','hyperparameter','accuracy','precision','recall','f1_score', 'mcc', 'auc','latency', 'fps','model size KB']
data_dynamic_quantized=[names_models]
data_dynamic_quantized.append(hyperparameters_values)
data_dynamic_quantized.extend(dynamic_quantized_all_results)
data_dynamic_quantized.append(tflite_dynamic_quantized_model_size)
print_table(data_dynamic_quantized,headers)
df_dynamic_quantized = export_table_to_files(data_dynamic_quantized, headers, filename_base="results_dynamic_quantized_weighted")

Confusion matrix:

In [ ]:
draw_confusion_matrix(confusion_dynamic_quantized_all[0:3],"SqueezeNet dynamic quantized Lite model", class_names=class_names)

## 2. Macro metrics 

In [ ]:
dynamic_quantized_macro_results, dynamic_quantized_predictions = \
    evaluate_tflite_quantized_models_macro(tflite_dynamic_quantized_model_files, test_ds)

headers=['models','accuracy','precision','recall','f1_score']
data_dynamic_quantized=[names_models]
data_dynamic_quantized.extend(dynamic_quantized_macro_results)
print_table(data_dynamic_quantized,headers)
df_dynamic_quantized = export_table_to_files(data_dynamic_quantized, headers, filename_base="results_dynamic_quantized_macro")

## 3. Micro metrics 

In [ ]:
dynamic_quantized_micro_results, dynamic_quantized_predictions = \
    evaluate_tflite_quantized_models_micro(tflite_dynamic_quantized_model_files, test_ds)

headers=['models','accuracy','precision','recall','f1_score']
data_dynamic_quantized=[names_models]
data_dynamic_quantized.extend(dynamic_quantized_micro_results)
print_table(data_dynamic_quantized,headers)
df_dynamic_quantized = export_table_to_files(data_dynamic_quantized, headers, filename_base="results_dynamic_quantized_micro")

## 4. Per-class metrics 

In [ ]:
dynamic_quantized_None_results, dynamic_quantized_predictions = \
    evaluate_tflite_quantized_models_None(tflite_dynamic_quantized_model_files, test_ds)

headers=['models','accuracy','precision','recall','f1_score']
data_dynamic_quantized=[names_models]
data_dynamic_quantized.extend(dynamic_quantized_None_results)
print_table(data_dynamic_quantized,headers)
df_dynamic_quantized = export_table_to_files(data_dynamic_quantized, headers, filename_base="results_dynamic_quantized_None")

## **Static Quantization**
The model is now smaller with quantized weights with some decrease in the accuracy, but other variable data are still in float format. 
Static quantization: To quantize variable data: intermediates between layers

In [ ]:
def representative_data_gen():
  for x, _ in train_ds.unbatch().batch(1).take(100):
        x = tf.cast(x, tf.float32)
        yield [x]

tflite_static_quantized_models = []

for i in range (0 , len(models_list)):
  converter = tf.lite.TFLiteConverter.from_keras_model(models_list[i])
  converter.optimizations = [tf.lite.Optimize.DEFAULT]
  converter.representative_dataset = representative_data_gen
  converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]

  # Convert the model to a quantized TFLite model
  tflite_static_quantized_models.append(converter.convert())

In [ ]:
tflite_static_quantized_model_files = []
tflite_static_quantized_model_size = []

base_filename_static = 'tflite_static_quantized_model'
# Create subfolder inside "converted_models"
path = tflite_models_dir/base_filename_static

if not os.path.exists(path):
  os.makedirs(path)

for i in range (1, len(models_list) + 1):
  file_name =  f"{base_filename_static}_{i}.tflite"
  tflite_static_quantized_model_files.append(path/file_name)
  tflite_static_quantized_model_files[i - 1].write_bytes(tflite_static_quantized_models[i - 1])
  tflite_static_quantized_model_size.append(os.path.getsize(tflite_static_quantized_model_files[i - 1]) / float(2**10))
  print("TFlite static quantized model in KB:", tflite_static_quantized_model_size[i-1])

# Evaluation of the Static Quantized Lite models

## 1. Weighted metrics 

In [ ]:
names_models = ['Static quantized SqueezeNet model 1','Static quantized SqueezeNet model 2', 'Static quantized SqueezeNet model 3']

In [ ]:
static_quantized_all_results, confusion_static_quantized_all, static_quantized_predictions = \
    evaluate_tflite_quantized_models_weighted(tflite_static_quantized_model_files, test_ds)

headers=['models','hyperparameter','accuracy','precision','recall','f1_score', 'mcc', 'auc','latency', 'fps','model size KB']
data_static_quantized=[names_models]
data_static_quantized.append(hyperparameters_values)
data_static_quantized.extend(static_quantized_all_results)
data_static_quantized.append(tflite_static_quantized_model_size)
print_table(data_static_quantized,headers)
df_static_quantized = export_table_to_files(data_static_quantized, headers, filename_base="results_static_quantized_weighted")

Draw confusion matrix

In [ ]:
draw_confusion_matrix(confusion_static_quantized_all[0:3],"SqueezeNet static quantized model", class_names=class_names)

## 2. Macro metrics 

In [ ]:
static_quantized_macro_results, static_quantized_predictions = \
    evaluate_tflite_quantized_models_macro(tflite_static_quantized_model_files, test_ds)

headers=['models','accuracy','precision','recall','f1_score']
data_static_quantized=[names_models]
data_static_quantized.extend(static_quantized_macro_results)
print_table(data_static_quantized,headers)
df_static_quantized = export_table_to_files(data_static_quantized, headers, filename_base="results_static_quantized_macro")

## 3. Micro metrics 

In [ ]:
static_quantized_micro_results, static_quantized_predictions = \
    evaluate_tflite_quantized_models_micro(tflite_static_quantized_model_files, test_ds)

headers=['models','accuracy','precision','recall','f1_score']
data_static_quantized=[names_models]
data_static_quantized.extend(static_quantized_micro_results)
print_table(data_static_quantized,headers)
df_static_quantized = export_table_to_files(data_static_quantized, headers, filename_base="results_static_quantized_micro")

## 4. Per-class metrics 

In [ ]:
static_quantized_None_results, static_quantized_predictions = \
    evaluate_tflite_quantized_models_None(tflite_static_quantized_model_files, test_ds)

headers=['models','accuracy','precision','recall','f1_score']
data_static_quantized=[names_models]
data_static_quantized.extend(static_quantized_None_results)
print_table(data_static_quantized,headers)
df_static_quantized = export_table_to_files(data_static_quantized, headers, filename_base="results_static_quantized_None")

## **Full static Quantization**
Now all weights and variable data are quantized. However, to maintain compatibility with floating-point models, the TensorFlow Lite Converter leaves the model input and output tensors in float. This is good for compatibility, but it won't be compatible with devices that perform only integer-based operations. 
Full static Quantization: End-to-end integer-only model.

In [ ]:
tflite_full_static_quantized_models = []

for i in range(0, len(models_list)):
  converter = tf.lite.TFLiteConverter.from_keras_model(models_list[i])
  converter.optimizations = [tf.lite.Optimize.DEFAULT]
  converter.representative_dataset = representative_data_gen
  converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
  converter.inference_input_type = tf.uint8
  converter.inference_output_type = tf.uint8
  tflite_full_static_quantized_models.append(converter.convert())

### The internal quantization remains the same as above, but the input and output tensors are now in integer format:

In [ ]:
interpreter = tf.lite.Interpreter(model_content = tflite_full_static_quantized_models[0])
input_type = interpreter.get_input_details()[0]['dtype']
print('input: ', input_type)
output_type = interpreter.get_output_details()[0]['dtype']
print('output: ', output_type)

In [ ]:
tflite_full_static_quantized_model_files = []
tflite_full_static_quantized_model_size = []

base_filename_full_static = 'tflite_full_static_quantized_model'
path = tflite_models_dir/base_filename_full_static

if not os.path.exists(path):
  os.makedirs(path)

for i in range(1, len(models_list) + 1) :
  file_name =  f"{base_filename_full_static}_{i}.tflite"
  tflite_full_static_quantized_model_files.append(path/file_name)
  tflite_full_static_quantized_model_files[i - 1].write_bytes(tflite_full_static_quantized_models[i - 1])
  tflite_full_static_quantized_model_size.append(os.path.getsize(tflite_full_static_quantized_model_files[i - 1]) / float(2**10))
  print("TFlite full static quantized model in KB:", tflite_full_static_quantized_model_size[i-1])

# Evaluation of the Full Static Quantized Lite models

## 1. Weighted metrics 

In [ ]:
names_models = ['Full static quantized SqueezeNet model 1','Full static quantized SqueezeNet model 2', 'Full static quantized SqueezeNet model 3']

In [ ]:
full_static_quantized_all_results, confusion_full_static_quantized_all, full_static_quantized_predictions = \
    evaluate_tflite_quantized_models_weighted(tflite_full_static_quantized_model_files, test_ds)

headers=['models','hyperparameter','accuracy','precision','recall','f1_score', 'mcc', 'auc','latency', 'fps','model size KB']
data_full_static_quantized=[names_models]
data_full_static_quantized.append(hyperparameters_values)
data_full_static_quantized.extend(full_static_quantized_all_results)
data_full_static_quantized.append(tflite_full_static_quantized_model_size)
print_table(data_full_static_quantized,headers)
df_full_static_quantized = export_table_to_files(data_full_static_quantized, headers, filename_base="results_full_static_quantized_weighted")

Draw confusion matrix

In [ ]:
draw_confusion_matrix(confusion_full_static_quantized_all[0:3],"SqueezeNet full static quantized model", class_names=class_names)

## 2. Macro metrics 

In [ ]:
full_static_quantized_macro_results, full_static_quantized_predictions = \
    evaluate_tflite_quantized_models_macro(tflite_full_static_quantized_model_files, test_ds)

headers=['models','accuracy','precision','recall','f1_score']
data_full_static_quantized=[names_models]
data_full_static_quantized.extend(full_static_quantized_macro_results)
print_table(data_full_static_quantized,headers)
df_full_static_quantized = export_table_to_files(data_full_static_quantized, headers, filename_base="results_full_static_quantized_macro")

## 3. Micro metrics 

In [ ]:
full_static_quantized_micro_results, full_static_quantized_predictions = \
    evaluate_tflite_quantized_models_micro(tflite_full_static_quantized_model_files, test_ds)

headers=['models','accuracy','precision','recall','f1_score']
data_full_static_quantized=[names_models]
data_full_static_quantized.extend(full_static_quantized_micro_results)
print_table(data_full_static_quantized,headers)
df_full_static_quantized = export_table_to_files(data_full_static_quantized, headers, filename_base="results_full_static_quantized_micro")

## 4. Per-class metrics 

In [ ]:
full_static_quantized_None_results, full_static_quantized_predictions = \
    evaluate_tflite_quantized_models_None(tflite_full_static_quantized_model_files, test_ds)

headers=['models','accuracy','precision','recall','f1_score']
data_full_static_quantized=[names_models]
data_full_static_quantized.extend(full_static_quantized_None_results)
print_table(data_full_static_quantized,headers)
df_full_static_quantized = export_table_to_files(data_full_static_quantized, headers, filename_base="results_full_static_quantized_None")

## Final comparison of the all models 

In [ ]:

headers = ["Model Type", "Accuracy", "Precision", "Recall", "F1 Score", "MCC", "AUC","Latency", "FPS", "Model Size (KB)"]

model_types = [
    "SqueezeNet model 1", "SqueezeNet model 2", "SqueezeNet model 3",
    "TFLite SqueezeNet model 1", "TFLite SqueezeNet model 2", "TFLite SqueezeNet model 3", 
    "Dynamic Quantized SqueezeNet model 1", "Dynamic Quantized SqueezeNet model 2", "Dynamic Quantized SqueezeNet model 3",  
    "Static Quantized SqueezeNet model 1", "Static Quantized SqueezeNet model 2", "Static Quantized SqueezeNet model 3", 
    "Full Static Quantized SqueezeNet model 1", "Full Static Quantized SqueezeNet model 2", "Full Static Quantized SqueezeNet model 3"
]

accuracies = (
    list(result_all_models_CNN[0]) +
    list(tflite_all_results[0]) +
    list(dynamic_quantized_all_results[0]) +
    list(static_quantized_all_results[0]) +
    list(full_static_quantized_all_results[0])
)

precisions = (
    list(result_all_models_CNN[1]) +
    list(tflite_all_results[1]) +
    list(dynamic_quantized_all_results[1]) +
    list(static_quantized_all_results[1]) +
    list(full_static_quantized_all_results[1])
)

recalls = (
    list(result_all_models_CNN[2]) +
    list(tflite_all_results[2]) +
    list(dynamic_quantized_all_results[2]) +
    list(static_quantized_all_results[2]) +
    list(full_static_quantized_all_results[2])
)

f1_scores = (
    list(result_all_models_CNN[3]) +
    list(tflite_all_results[3]) +
    list(dynamic_quantized_all_results[3]) +
    list(static_quantized_all_results[3])+
    list(full_static_quantized_all_results[3])
)

mcc_scores = (
    list(result_all_models_CNN[4]) +
    list(tflite_all_results[4]) +
    list(dynamic_quantized_all_results[4]) +
    list(static_quantized_all_results[4]) +
    list(full_static_quantized_all_results[4])
)

auc_scores = (
    list(result_all_models_CNN[5]) +
    list(tflite_all_results[5]) +
    list(dynamic_quantized_all_results[5]) +
    list(static_quantized_all_results[5]) +
    list(full_static_quantized_all_results[5])
)

latencies = (
    list(result_all_models_CNN[6]) +
    list(tflite_all_results[6]) +
    list(dynamic_quantized_all_results[6]) +
    list(static_quantized_all_results[6]) +
    list(full_static_quantized_all_results[6])
)

fps_values = (
    list(result_all_models_CNN[7]) +
    list(tflite_all_results[7]) +
    list(dynamic_quantized_all_results[7]) +
    list(static_quantized_all_results[7]) +
    list(full_static_quantized_all_results[7])
)

sizes = (
    list(model_size_kb) +
    list(tflite_model_size) +
    list(tflite_dynamic_quantized_model_size) +
    list(tflite_static_quantized_model_size) +
    list(tflite_full_static_quantized_model_size)
)

final_data = [
    model_types,
    accuracies,
    precisions,
    recalls,
    f1_scores,
    mcc_scores,
    auc_scores,
    latencies,
    fps_values,
    sizes,
]

print_table(final_data, headers)
df_final_data = export_table_to_files(final_data, headers, filename_base="results_final_data")


# Explainability and model interpretation:

### LIME (Local Interpretable Model-agnostic Explanation) explanation for one random test image per class

Reference: https://github.com/marcotcr/lime/blob/master/doc/notebooks/Tutorial%20-%20images%20-%20Pytorch.ipynb

In [ ]:
best_cnn_idx = int(np.argmax(result_all_models_CNN[0]))
lime_model = models_list_CNN[best_cnn_idx]
lime_model_name = model_names_CNN[best_cnn_idx]
print(f'Using LIME with: {lime_model_name}')

def lime_predict_fn(images):
    images = np.asarray(images, dtype=np.float32)
    if images.max() > 1.0:
        images = images / 255.0
    return lime_model.predict(images, verbose=0)

def explain_random_image_per_class(paths, labels, class_names, predict_fn, img_size, plot_dir,
                                   num_samples=2000, num_features=8,
                                   positive_only=True, hide_rest=False,
                                   random_state=1):
    explainer = lime_image.LimeImageExplainer(random_state=random_state)
    rng = np.random.default_rng(random_state)
    n_classes = len(class_names)
    rows = 2
    cols = math.ceil(n_classes / rows)
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes = np.atleast_1d(axes).ravel()
    selected_rows = []

    for class_idx, class_name in enumerate(class_names):
        class_indices = np.where(labels == class_idx)[0]
        if len(class_indices) == 0:
            axes[class_idx].axis('off')
            continue

        sample_idx = int(rng.choice(class_indices))
        image_path = paths[sample_idx]
        image = tf.keras.utils.load_img(image_path, target_size=img_size)
        image_np = tf.keras.utils.img_to_array(image).astype(np.float32) / 255.0

        explanation = explainer.explain_instance(
            image_np,
            classifier_fn=predict_fn,
            top_labels=1,
            hide_color=0,
            num_samples=num_samples,
        )

        pred_proba = predict_fn(np.expand_dims(image_np, axis=0))[0]
        pred_idx = int(np.argmax(pred_proba))
        pred_name = class_names[pred_idx]
        pred_prob = float(pred_proba[pred_idx])

        temp, mask = explanation.get_image_and_mask(
            explanation.top_labels[0],
            positive_only=positive_only,
            num_features=num_features,
            hide_rest=hide_rest,
        )

        axes[class_idx].imshow(mark_boundaries(np.clip(temp, 0, 1), mask))
        axes[class_idx].set_title(f'True: {class_name}\nPred: {pred_name} ({pred_prob:.2f})', fontsize=10)
        axes[class_idx].axis('off')

        selected_rows.append({
            'class_name': class_name,
            'image_path': image_path,
            'predicted_class': pred_name,
            'predicted_probability': pred_prob,
        })

    for ax in axes[n_classes:]:
        ax.axis('off')

    plt.suptitle(f'LIME explanations with {lime_model_name}', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(plot_dir, 'LIME_random_test_image_per_class_2.png'), dpi=300, bbox_inches='tight')
    plt.show()

    return pd.DataFrame(selected_rows)

lime_examples_df = explain_random_image_per_class(
    X_test,
    y_test,
    class_names,
    lime_predict_fn,
    img_size,
    PLOT_DIR,
    num_samples=2000,
    num_features=8,
    positive_only=True,
    hide_rest=False,
    random_state=1,
)

lime_examples_df